# TabTransformer 시도

대회 규칙에서 "사전학습 모델 사용 가능"이 명시되어 있고, 리더보드 상위팀(0.742대)과
우리(0.7400)의 격차를 설명할 단서로 의심되는 부분입니다. LightGBM 계열에서는
거의 모든 추가 시도(교차피처, 결측조합, 모델앙상블, 부스팅타입 변경, 클래스불균형
처리 등)가 효과가 없었는데, 이는 "전통적 그래디언트 부스팅"의 천장에 도달했다는
신호일 수 있습니다. TabTransformer는 범주형 변수를 임베딩 + Self-Attention으로
처리하는 딥러닝 기반 테이블 모델로, LightGBM과는 본질적으로 다른 방식으로 패턴을
학습하기 때문에 시도해볼 가치가 있습니다.

**주의**: 이 노트북은 CPU에서도 동작하지만, GPU가 없으면 epoch당 시간이 오래
걸립니다 (이 환경에서 1코어 CPU 기준 약 95~100초/epoch). 노트북(M-series Mac
포함)은 보통 이보다 빠릅니다. 처음 2 epoch 실행 시간을 보고 전체 시간을
가늠해보세요.

**사전 준비**: `uv add torch`


## 1. 라이브러리 및 설정

In [1]:
import os
import time
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler

warnings.filterwarnings("ignore")
torch.manual_seed(1004)

t0 = time.time()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

DATA_DIR = "../data"
TRAIN_PATH = f"{DATA_DIR}/train.csv"
TARGET_COL = "임신 성공 여부"
DEAD_COLS = ["불임 원인 - 여성 요인", "난자 채취 경과일"]

N_SPLITS = 5  # 시간이 오래 걸리면 3으로 낮춰도 괜찮음
EPOCHS = 15   # LightGBM 수준(0.739) 근처까지는 보통 10epoch 전후로 수렴하는 경향을 보였음
CHECKPOINT_DIR = "tt_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

device: cpu


## 2. 데이터 로드 및 전처리 (main.py와 동일한 베이스)

In [2]:
train_raw = pd.read_csv(TRAIN_PATH).drop(columns=["ID"])


def base_features(df):
    df = df.copy()
    df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
    df["froze_embryo"] = df["동결 배아 사용 여부"].fillna(0).astype(int)
    return df


def fill_na(df):
    cat_cols = df.select_dtypes(include=["object"]).columns
    num_cols = df.select_dtypes(include=["int64", "float64"]).columns
    na_cols = df.isna().sum().loc[lambda x: x > 0].index
    for col in na_cols:
        if col in cat_cols:
            df[col] = df[col].fillna("해당없음")
        elif col in num_cols:
            df[col] = df[col].fillna(-1)
    return df


df = fill_na(base_features(train_raw.copy()).drop(columns=DEAD_COLS))
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
num_cols = [c for c in df.columns if c not in cat_cols and c != TARGET_COL]

# 범주형: LabelEncoder (TabTransformer 임베딩 입력용 정수 인덱스)
cat_cardinalities = []
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    cat_cardinalities.append(df[col].nunique())

y = df[TARGET_COL].values.astype(np.float32)
X_cat = df[cat_cols].values.astype(np.int64)
X_num = df[num_cols].values.astype(np.float32)

print(f"전처리 완료: cat_cols={len(cat_cols)}, num_cols={len(num_cols)}, rows={len(df)}")

전처리 완료: cat_cols=20, num_cols=47, rows=256351


## 3. TabTransformer 모델 정의

In [3]:
class TabTransformer(nn.Module):
    def __init__(self, cat_cardinalities, num_numeric, dim=32, depth=3, heads=4, dropout=0.1):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(card + 1, dim) for card in cat_cardinalities])
        self.n_cat = len(cat_cardinalities)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=heads, dim_feedforward=dim * 4, dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)

        self.num_norm = nn.LayerNorm(num_numeric)

        mlp_input_dim = self.n_cat * dim + num_numeric
        self.mlp = nn.Sequential(
            nn.Linear(mlp_input_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, x_cat, x_num):
        embs = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
        embs = torch.stack(embs, dim=1)  # (batch, n_cat, dim)
        embs = self.transformer(embs)  # self-attention across categorical fields
        embs_flat = embs.reshape(embs.size(0), -1)

        x_num_norm = self.num_norm(x_num)
        combined = torch.cat([embs_flat, x_num_norm], dim=1)
        return self.mlp(combined).squeeze(-1)

## 4. 학습 함수 (체크포인트 지원)

epoch 단위로 체크포인트를 저장합니다. 너무 오래 걸려서 중간에 멈춰도, 같은 셀을
다시 실행하면 멈췄던 지점부터 이어서 학습합니다 (fold별로 체크포인트 파일이
따로 저장됩니다).


In [4]:
def train_tabtransformer(X_cat_tr, X_num_tr, y_tr, X_cat_val, X_num_val, y_val, cat_cardinalities,
                          epochs, checkpoint_path):
    scaler = StandardScaler()
    X_num_tr_scaled = scaler.fit_transform(X_num_tr).astype(np.float32)
    X_num_val_scaled = scaler.transform(X_num_val).astype(np.float32)

    model = TabTransformer(cat_cardinalities, num_numeric=X_num_tr.shape[1]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    criterion = nn.BCEWithLogitsLoss()

    start_epoch = 0
    best_auc = 0.0
    if os.path.exists(checkpoint_path):
        ckpt = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        start_epoch = ckpt["epoch"]
        best_auc = ckpt["best_auc"]
        print(f"  체크포인트 로드: epoch {start_epoch}부터 이어서 시작 (best_auc={best_auc:.5f})")

    # 데이터는 numpy 상태로 유지하고, 배치마다 필요한 부분만 텐서로 변환 (메모리 절약)
    n_samples = len(y_tr)
    batch_size = 1024
    val_batch_size = 4096
    best_pred = None
    patience_counter = 0

    for epoch in range(start_epoch, epochs):
        model.train()
        perm = np.random.permutation(n_samples)
        total_loss = 0.0
        for i in range(0, n_samples, batch_size):
            idx = perm[i : i + batch_size]
            x_cat_batch = torch.from_numpy(X_cat_tr[idx]).long().to(device)
            x_num_batch = torch.from_numpy(X_num_tr_scaled[idx]).float().to(device)
            y_batch = torch.from_numpy(y_tr[idx]).float().to(device)

            optimizer.zero_grad()
            out = model(x_cat_batch, x_num_batch)
            loss = criterion(out, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(idx)

        model.eval()
        val_preds = []
        with torch.no_grad():
            for i in range(0, len(y_val), val_batch_size):
                x_cat_v = torch.from_numpy(X_cat_val[i : i + val_batch_size]).long().to(device)
                x_num_v = torch.from_numpy(X_num_val_scaled[i : i + val_batch_size]).float().to(device)
                logits = model(x_cat_v, x_num_v)
                val_preds.append(torch.sigmoid(logits).cpu().numpy())
        val_pred = np.concatenate(val_preds)
        val_auc = roc_auc_score(y_val, val_pred)

        print(f"  [{time.time()-t0:.1f}s] epoch {epoch+1}/{epochs}  loss={total_loss/n_samples:.4f}  val_auc={val_auc:.5f}")

        improved = val_auc > best_auc
        if improved:
            best_auc = val_auc
            best_pred = val_pred
            patience_counter = 0
        else:
            patience_counter += 1

        torch.save(
            {
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "epoch": epoch + 1,
                "best_auc": best_auc,
            },
            checkpoint_path,
        )

        if patience_counter >= 4:
            print("  early stopping (4 epoch 연속 개선 없음)")
            break

    return best_auc, best_pred

## 5. K-Fold 학습 실행

각 fold마다 별도 체크포인트 파일(`tt_checkpoints/fold{N}.pt`)을 사용합니다.
이 셀을 여러 번 실행해도 이미 끝난 fold는 건너뛰고, 진행 중인 fold는 이어서
계속됩니다.


In [5]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=1004)
fold_scores = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_cat, y), start=1):
    checkpoint_path = f"{CHECKPOINT_DIR}/fold{fold}.pt"

    # 이미 끝난(=epochs까지 다 돌았거나 early stop한) fold는 점수만 출력하고 스킵
    if os.path.exists(checkpoint_path):
        ckpt = torch.load(checkpoint_path, map_location=device)
        if ckpt["epoch"] >= EPOCHS:
            print(f"Fold {fold}: 이미 완료됨 (best_auc={ckpt['best_auc']:.5f}), 스킵")
            fold_scores.append(ckpt["best_auc"])
            continue

    print(f"\n=== Fold {fold}/{N_SPLITS} ===")
    auc, _ = train_tabtransformer(
        X_cat[tr_idx], X_num[tr_idx], y[tr_idx],
        X_cat[val_idx], X_num[val_idx], y[val_idx],
        cat_cardinalities, epochs=EPOCHS, checkpoint_path=checkpoint_path,
    )
    fold_scores.append(auc)
    print(f"Fold {fold} best AUC: {auc:.5f}")

print(f"\n{'='*50}")
print(f"TabTransformer {N_SPLITS}-Fold 평균: {np.mean(fold_scores):.5f}  (fold별: {[round(s,4) for s in fold_scores]})")
print(f"비교 대상 (현재 main.py, LightGBM 5-Fold): 0.7393~0.7400")

Fold 1: 이미 완료됨 (best_auc=0.73866), 스킵

=== Fold 2/5 ===
  체크포인트 로드: epoch 14부터 이어서 시작 (best_auc=0.73671)
  [32.1s] epoch 15/15  loss=0.4881  val_auc=0.73497
Fold 2 best AUC: 0.73671
Fold 3: 이미 완료됨 (best_auc=0.73464), 스킵
Fold 4: 이미 완료됨 (best_auc=0.73457), 스킵
Fold 5: 이미 완료됨 (best_auc=0.73640), 스킵

TabTransformer 5-Fold 평균: 0.73620  (fold별: [0.7387, 0.7367, 0.7346, 0.7346, 0.7364])
비교 대상 (현재 main.py, LightGBM 5-Fold): 0.7393~0.7400


## 6. 결론 정리

이 셀은 모든 fold가 끝난 뒤(또는 충분히 많은 epoch이 지난 뒤) 실행해서, LightGBM과
점수를 비교합니다. TabTransformer가 LightGBM을 능가한다면 본격적으로 test 예측
및 제출 파일 생성으로 이어가면 됩니다. 그렇지 않다면, "사전학습 모델"이 격차의
원인이 아니었다는 것을 시사하므로 다른 가설(피처 구성, 검증되지 않은 외부 정보 등)을
검토하는 게 좋습니다.


In [6]:
lgbm_score = 0.7393  # main.py 기준 (참고용, 직접 갱신 가능)
tt_score = np.mean(fold_scores)

print(f"LightGBM (5-Fold):     {lgbm_score:.5f}")
print(f"TabTransformer ({N_SPLITS}-Fold): {tt_score:.5f}")
print(f"차이: {tt_score - lgbm_score:+.5f}")

if tt_score > lgbm_score:
    print("\nTabTransformer가 더 우세합니다 - test 예측 및 제출 파일 생성으로 진행 가능")
else:
    print("\nLightGBM이 여전히 더 우세하거나 비슷합니다 - 사전학습 모델이 격차의 원인은 아닐 가능성")

LightGBM (5-Fold):     0.73930
TabTransformer (5-Fold): 0.73620
차이: -0.00310

LightGBM이 여전히 더 우세하거나 비슷합니다 - 사전학습 모델이 격차의 원인은 아닐 가능성
